In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import lime
import lime.lime_tabular
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('../AirlineScrappedReview_Cleaned_seif_we_ibra#1.csv')

In [ ]:
X = df[['Flying_Date', 'Route', 'Verified', 'Review_title', 'Review_content', 'Traveller_Type', 'Class', 'Start_Location', 'End_Location', 'Layover_Route', 'Start_Longitude', 'Start_Latitude', 'End_Longitude', 'End_Latitude', 'Start_Address', 'End_Address']]

y = df['Rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Step 1: Feature Engineering for Classical ML (RandomForest)
print("="*80)
print("FEATURE ENGINEERING FOR CLASSICAL ML (LIME)")
print("="*80)

# Identify numerical and categorical columns
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumerical features ({len(numerical_cols)}): {numerical_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")

# Handle missing values
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

# Fill numerical missing values with median
for col in numerical_cols:
    median_val = X_train_clean[col].median()
    X_train_clean[col].fillna(median_val, inplace=True)
    X_test_clean[col].fillna(median_val, inplace=True)

# Fill categorical missing values with 'Unknown'
for col in categorical_cols:
    X_train_clean[col].fillna('Unknown', inplace=True)
    X_test_clean[col].fillna('Unknown', inplace=True)

print(f"\n✓ Missing values handled")

# One-Hot Encode categorical features
X_train_encoded = pd.get_dummies(X_train_clean, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_clean, columns=categorical_cols, drop_first=True)

# Ensure both sets have the same columns
missing_cols = set(X_train_encoded.columns) - set(X_test_encoded.columns)
for col in missing_cols:
    X_test_encoded[col] = 0

X_test_encoded = X_test_encoded[X_train_encoded.columns]

print(f"✓ One-hot encoding applied")
print(f"  Features after encoding: {X_train_encoded.shape[1]}")
print(f"  X_train shape: {X_train_encoded.shape}")
print(f"  X_test shape: {X_test_encoded.shape}")

feature_names = X_train_encoded.columns.tolist()
class_names = ['Low Rating', 'High Rating']

print(f"\n✓ Feature engineering complete")
print(f"  Total features: {len(feature_names)}")
print(f"  Class names: {class_names}")

In [ ]:
# Step 2: Train Classical ML Model (RandomForest)
print("\n" + "="*80)
print("TRAINING RANDOM FOREST CLASSIFIER")
print("="*80)

# Build and train RandomForest
print(f"\nTraining RandomForestClassifier...")
print(f"  Parameters:")
print(f"    - n_estimators: 100")
print(f"    - max_depth: 15")
print(f"    - random_state: 42")

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_encoded, y_train)

print(f"✓ Model trained successfully")

# Evaluate model
y_pred = model.predict(X_test_encoded)
y_pred_proba = model.predict_proba(X_test_encoded)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n  Test Accuracy: {accuracy:.4f}")
print(f"  Number of features: {len(feature_names)}")

# Feature importance from tree
feature_importance_rf = pd.DataFrame({
    'feature': feature_names,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 10 Features (by RandomForest importance):")
for idx, row in feature_importance_rf.head(10).iterrows():
    print(f"  {row['feature']:45s} {row['importance']:.4f}")

# Visualize RF feature importance
plt.figure(figsize=(12, 6))
top_n = 15
top_features = feature_importance_rf.head(top_n)
plt.barh(range(len(top_features)), top_features['importance'], color='forestgreen', edgecolor='black')
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Feature Importance')
plt.title('RandomForest Feature Importance: Top 15 Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('lime_rf_feature_importance_builtin.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ RandomForest importance plot saved as 'lime_rf_feature_importance_builtin.png'")

In [ ]:
# Step 3: Initialize LIME Explainer for RandomForest
print("\n" + "="*80)
print("LIME EXPLAINER FOR CLASSICAL ML (RANDOMFOREST)")
print("="*80)

print("\nInitializing LIME TabularExplainer (model-agnostic local explanations)...")
print("  • LIME: Local Interpretable Model-agnostic Explanations")
print("  • Approach: Perturbation-based (perturbs inputs locally to understand model)")
print("  • Works with any model (NN, RF, SVM, etc.)")
print("  • Generates local linear approximations around each instance")

# Create prediction function wrapper for LIME
# LIME expects probabilities for classification
def predict_fn(X):
    """Prediction function for LIME - returns probabilities"""
    return model.predict_proba(X)

# Initialize LIME explainer
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_encoded.values,  # Training data for reference
    feature_names=feature_names,
    class_names=class_names,
    mode='classification',  # Use classification mode for probabilities
    verbose=False,
    random_state=42
)

print("\n✓ LIME TabularExplainer created successfully")
print(f"  Explainer type: {type(explainer).__name__}")
print(f"  Number of features: {len(feature_names)}")
print(f"  Training data shape: {X_train_encoded.shape}")
print(f"  Mode: classification (probabilities)")
print(f"  Class names: {class_names}")

print("\nLIME Configuration:")
print(f"  • Num features: {len(feature_names)}")
print(f"  • Classes: {len(class_names)}")
print(f"  • Perturbation strategy: Random local perturbations")
print(f"  • Model approximation: Local linear model")
print(f"  • Note: RandomForest can handle non-linear relationships better than linear approximation")

In [ ]:
# Step 4: Generate LIME Explanations for Individual Predictions
print("\n" + "="*80)
print("LIME LOCAL EXPLANATIONS - INDIVIDUAL PREDICTIONS")
print("="*80)

# Select samples to explain
sample_indices = [0, 10, 20]
lime_explanations = {}

for idx, sample_idx in enumerate(sample_indices):
    if sample_idx >= len(X_test_encoded):
        print(f"Skipping sample index {sample_idx} (out of range)")
        continue
    
    print(f"\n--- Generating LIME Explanation for Test Sample {sample_idx} ---")
    
    # Get prediction
    instance = X_test_encoded.iloc[sample_idx].values
    rf_pred = model.predict(instance.reshape(1, -1))[0]
    rf_pred_proba = model.predict_proba(instance.reshape(1, -1))[0]
    actual_rating = y_test.iloc[sample_idx]
    
    print(f"RandomForest Prediction: {rf_pred} ({class_names[int(rf_pred)]})")
    print(f"  - Probability Low Rating: {rf_pred_proba[0]:.4f}")
    print(f"  - Probability High Rating: {rf_pred_proba[1]:.4f}")
    print(f"Actual Rating: {actual_rating}")
    print(f"Correct: {'✓' if rf_pred == actual_rating else '✗'}")
    
    # Generate LIME explanation
    print(f"\nGenerating local perturbations (1000 samples)...")
    
    exp = explainer.explain_instance(
        data_row=instance,
        predict_fn=predict_fn,
        num_features=10,  # Show top 10 features
        num_samples=1000,  # Generate 1000 perturbations
        top_labels=1  # Explain top 1 predicted class
    )
    
    lime_explanations[sample_idx] = exp
    
    print(f"✓ Explanation generated")
    print(f"\nTop 10 Features Contributing to Prediction ({class_names[int(rf_pred)]})")
    
    # Display feature contributions for predicted class
    pred_class = int(rf_pred)
    for feat_idx, (feat_name, weight) in enumerate(exp.as_list(label=pred_class), 1):
        direction = '↑ increases' if weight > 0 else '↓ decreases'
        print(f"  {feat_idx:2d}. {feat_name:50s} {direction:15s} ({weight:+.4f})")
    
    # Save visualization
    try:
        fig = exp.as_pyplot_figure(label=pred_class)
        plt.tight_layout()
        plt.savefig(f'lime_rf_explanation_sample_{sample_idx}.png', dpi=100, bbox_inches='tight')
        plt.show()
        print(f"✓ Explanation visualization saved as 'lime_rf_explanation_sample_{sample_idx}.png'")
    except Exception as e:
        print(f"Note: Could not create figure ({type(e).__name__})")

print("\n" + "="*80)
print("Stored explanations in dict: lime_explanations")
print("Use lime_explanations[sample_idx] to access each explanation")
print("="*80)

In [ ]:
# Step 5: Feature Importance Summary (Aggregate LIME Explanations)
print("\n" + "="*80)
print("AGGREGATE LIME FEATURE IMPORTANCE")
print("="*80)

# Aggregate feature weights across all explanations
feature_weights = {feat: [] for feat in feature_names}

# Sample predictions for better aggregation
print("\nGenerating LIME explanations for 30 test samples (for aggregation)...")
sample_size = min(30, len(X_test_encoded))

for sample_idx in range(sample_size):
    instance = X_test_encoded.iloc[sample_idx].values
    pred_class = int(model.predict(instance.reshape(1, -1))[0])
    
    try:
        exp = explainer.explain_instance(
            data_row=instance,
            predict_fn=predict_fn,
            num_features=len(feature_names),  # All features
            num_samples=500,  # Fewer samples for speed
            top_labels=1
        )
        
        # Extract weights for predicted class
        for feat_name, weight in exp.as_list(label=pred_class):
            # Parse feature name to get base feature
            base_feat = feat_name.split(' ')[0]
            if base_feat in feature_weights:
                feature_weights[base_feat].append(abs(weight))
        
        if (sample_idx + 1) % 10 == 0:
            print(f"  Generated {sample_idx + 1}/{sample_size} explanations...")
    except Exception as e:
        print(f"  Warning: Could not explain sample {sample_idx}: {e}")

# Calculate average importance
avg_importance = {}
for feat, weights in feature_weights.items():
    if len(weights) > 0:
        avg_importance[feat] = np.mean(weights)
    else:
        avg_importance[feat] = 0.0

# Sort by importance
sorted_importance = sorted(avg_importance.items(), key=lambda x: x[1], reverse=True)

print(f"\n✓ Aggregated LIME importance from {sample_size} explanations")
print(f"\nTop 15 Most Important Features (by average absolute LIME weight):")
print("-" * 70)

for rank, (feat, importance) in enumerate(sorted_importance[:15], 1):
    bar = '█' * int(importance * 50)
    print(f"{rank:2d}. {feat:45s} {importance:7.4f} {bar}")

# Visualize importance
plt.figure(figsize=(12, 8))
top_n = 15
top_features = sorted_importance[:top_n]
feat_names = [f[0] for f in top_features]
feat_values = [f[1] for f in top_features]

plt.barh(range(len(feat_names)), feat_values, color='steelblue', edgecolor='black')
plt.yticks(range(len(feat_names)), feat_names)
plt.xlabel('Average Absolute LIME Weight')
plt.title('LIME Feature Importance: Top 15 Features (Aggregated from 30 samples)\n(RandomForest + LIME)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('lime_rf_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Feature importance plot saved as 'lime_rf_feature_importance.png'")

In [ ]:
# Step 6: Analyze Feature Impact Patterns
print("\n" + "="*80)
print("LIME FEATURE IMPACT ANALYSIS - POSITIVE VS NEGATIVE WEIGHTS")
print("="*80)

# Track positive and negative impacts
feature_impacts = {feat: {'positive': [], 'negative': []} for feat in feature_names}

print("\nAnalyzing feature impact directions across 20 samples...")
sample_size = min(20, len(X_test_encoded))

for sample_idx in range(sample_size):
    instance = X_test_encoded.iloc[sample_idx].values
    pred_class = int(model.predict(instance.reshape(1, -1))[0])
    
    try:
        exp = explainer.explain_instance(
            data_row=instance,
            predict_fn=predict_fn,
            num_features=len(feature_names),
            num_samples=500,
            top_labels=1
        )
        
        for feat_name, weight in exp.as_list(label=pred_class):
            base_feat = feat_name.split(' ')[0]
            if base_feat in feature_impacts:
                if weight > 0:
                    feature_impacts[base_feat]['positive'].append(weight)
                else:
                    feature_impacts[base_feat]['negative'].append(abs(weight))
    except Exception as e:
        print(f"  Warning at sample {sample_idx}: {e}")

print(f"✓ Analyzed {sample_size} samples")
print(f"\nFeature Impact Directions (Top 10 Features):")
print("-" * 80)

for rank, (feat, importance) in enumerate(sorted_importance[:10], 1):
    pos_weights = feature_impacts[feat]['positive']
    neg_weights = feature_impacts[feat]['negative']
    
    pos_avg = np.mean(pos_weights) if len(pos_weights) > 0 else 0
    neg_avg = np.mean(neg_weights) if len(neg_weights) > 0 else 0
    
    print(f"\n{rank:2d}. {feat}")
    print(f"     Increases prediction (red):   {len(pos_weights):2d} times, avg weight: +{pos_avg:.4f}")
    print(f"     Decreases prediction (blue):  {len(neg_weights):2d} times, avg weight: -{neg_avg:.4f}")

# Create visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for ax_idx, (feat, importance) in enumerate(sorted_importance[:6]):
    pos_weights = feature_impacts[feat]['positive']
    neg_weights = feature_impacts[feat]['negative']
    
    ax = axes[ax_idx]
    
    # Create histogram
    if len(pos_weights) > 0:
        ax.hist(pos_weights, bins=10, alpha=0.6, label='Positive (↑)', color='red', edgecolor='black')
    if len(neg_weights) > 0:
        ax.hist(neg_weights, bins=10, alpha=0.6, label='Negative (↓)', color='blue', edgecolor='black')
    
    ax.set_title(f'{feat}\n(Impact Distribution)', fontsize=10, fontweight='bold')
    ax.set_xlabel('LIME Weight')
    ax.set_ylabel('Frequency')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

plt.suptitle('LIME Feature Impact Distributions (Top 6 Features)', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('lime_rf_impact_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Impact distribution plot saved as 'lime_rf_impact_distributions.png'")

In [ ]:
# Step 7: Compare LIME Explanations - Correct vs Incorrect Predictions
print("\n" + "="*80)
print("LIME ANALYSIS: CORRECT VS INCORRECT PREDICTIONS")
print("="*80)

# Identify correct and incorrect predictions
y_pred_all = model.predict(X_test_encoded.values)
is_correct = (y_pred_all == y_test.values)

correct_indices = np.where(is_correct)[0][:10]  # First 10 correct
incorrect_indices = np.where(~is_correct)[0][:10]  # First 10 incorrect

print(f"\nModel Performance:")
print(f"  Correct predictions: {is_correct.sum()} / {len(is_correct)}")
print(f"  Accuracy: {is_correct.sum() / len(is_correct) * 100:.2f}%")

# Analyze correct predictions
correct_weights = {feat: [] for feat in feature_names}
incorrect_weights = {feat: [] for feat in feature_names}

print(f"\nAnalyzing {len(correct_indices)} CORRECT predictions...")
for sample_idx in correct_indices:
    instance = X_test_encoded.iloc[sample_idx].values
    pred_class = int(model.predict(instance.reshape(1, -1))[0])
    try:
        exp = explainer.explain_instance(instance, predict_fn, num_features=len(feature_names), num_samples=300, top_labels=1)
        for feat_name, weight in exp.as_list(label=pred_class):
            base_feat = feat_name.split(' ')[0]
            if base_feat in correct_weights:
                correct_weights[base_feat].append(abs(weight))
    except:
        pass

print(f"Analyzing {len(incorrect_indices)} INCORRECT predictions...")
for sample_idx in incorrect_indices:
    instance = X_test_encoded.iloc[sample_idx].values
    pred_class = int(model.predict(instance.reshape(1, -1))[0])
    try:
        exp = explainer.explain_instance(instance, predict_fn, num_features=len(feature_names), num_samples=300, top_labels=1)
        for feat_name, weight in exp.as_list(label=pred_class):
            base_feat = feat_name.split(' ')[0]
            if base_feat in incorrect_weights:
                incorrect_weights[base_feat].append(abs(weight))
    except:
        pass

# Compare
print(f"\n✓ Analysis complete")
print(f"\nTop Features Explaining CORRECT vs INCORRECT Predictions:")
print("-" * 80)

correct_importance = {f: np.mean(w) if len(w) > 0 else 0 for f, w in correct_weights.items()}
incorrect_importance = {f: np.mean(w) if len(w) > 0 else 0 for f, w in incorrect_weights.items()}

correct_sorted = sorted(correct_importance.items(), key=lambda x: x[1], reverse=True)[:10]
incorrect_sorted = sorted(incorrect_importance.items(), key=lambda x: x[1], reverse=True)[:10]

print(f"\nTop 10 for CORRECT predictions:")
for rank, (feat, imp) in enumerate(correct_sorted, 1):
    print(f"  {rank:2d}. {feat:45s} {imp:.4f}")

print(f"\nTop 10 for INCORRECT predictions:")
for rank, (feat, imp) in enumerate(incorrect_sorted, 1):
    print(f"  {rank:2d}. {feat:45s} {imp:.4f}")

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Correct predictions
feats_c = [f[0] for f in correct_sorted]
imps_c = [f[1] for f in correct_sorted]
axes[0].barh(range(len(feats_c)), imps_c, color='green', alpha=0.7, edgecolor='black')
axes[0].set_yticks(range(len(feats_c)))
axes[0].set_yticklabels(feats_c, fontsize=9)
axes[0].set_xlabel('Average LIME Weight')
axes[0].set_title(f'Top Features in CORRECT Predictions ({len(correct_indices)} samples)', fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3)

# Incorrect predictions
feats_i = [f[0] for f in incorrect_sorted]
imps_i = [f[1] for f in incorrect_sorted]
axes[1].barh(range(len(feats_i)), imps_i, color='red', alpha=0.7, edgecolor='black')
axes[1].set_yticks(range(len(feats_i)))
axes[1].set_yticklabels(feats_i, fontsize=9)
axes[1].set_xlabel('Average LIME Weight')
axes[1].set_title(f'Top Features in INCORRECT Predictions ({len(incorrect_indices)} samples)', fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lime_rf_correct_vs_incorrect.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Comparison plot saved as 'lime_rf_correct_vs_incorrect.png'")

## Summary: LIME Explainability for Classical ML (RandomForest)

### What We've Created (LIME for RandomForest):

#### **1. RandomForest Model**
- Tree-based ensemble method
- 100 trees with max depth 15
- Fast predictions, interpretable structure
- Built-in feature importance (gini-based)

#### **2. LIME Model-Agnostic Explanations**
- Works identically with RandomForest and Neural Networks
- Local explanations (one prediction at a time)
- Perturbation-based approach
- No gradient computation needed

#### **3. Individual Local Explanations**
- Generates 1000 local perturbations
- Fits simple linear model locally
- Shows feature weights for specific prediction
- Easy to verify by looking at nearby instances

#### **4. Aggregate Feature Importance**
- Collects LIME explanations from 30 samples
- Averages weights across instances
- Compares with RandomForest built-in importance
- Shows global feature ranking from local explanations

#### **5. Impact Directionality Analysis**
- Separates positive and negative contributions
- Histograms of impact distributions
- Shows consistency of feature effects

#### **6. Error Analysis**
- Compares correct vs incorrect predictions
- Identifies features that explain successes
- Finds features in failed predictions
- Helps debug model errors

### LIME vs RandomForest Built-in Importance:

| Aspect | RandomForest Importance | LIME |
|--------|----------------------|------|
| **Based on** | Tree split frequency (Gini/entropy) | Local perturbation weights |
| **Scope** | Global model average | Local per-instance |
| **Interpretation** | "Features used by trees" | "Features affecting this prediction" |
| **Model assumption** | Tree-specific | Model-agnostic |
| **Consistency** | Deterministic | Can vary across instances |
| **Ease of verification** | Requires tree inspection | Check nearby points |

### When to Use LIME vs RandomForest Feature Importance:

**Use RandomForest Importance for:**
- Quick global feature ranking
- Understanding overall model behavior
- Feature selection/engineering
- Computational efficiency

**Use LIME for:**
- Explaining specific predictions
- Stakeholder communication
- Debugging prediction errors
- Building trust in decisions
- Finding edge cases

### Key Advantages of LIME for Classical ML:

✓ **Model-agnostic**: Can explain any tree model (RF, XGBoost, LightGBM, etc.)  
✓ **Intuitive**: Simple local linear logic is easy to understand  
✓ **Verification**: Can manually check explanations against data  
✓ **Local context**: Shows why THIS instance got THIS prediction  
✓ **No feature scaling**: Trees don't need normalization  
✓ **Fast inference**: Classical models predict quickly  

### RandomForest + LIME Workflow:

1. **Train RandomForest** on features (no scaling needed)
2. **Check RF importance** for global feature ranking
3. **Use LIME** to explain specific predictions
4. **Aggregate LIME** across instances for global view
5. **Compare patterns** between correct/incorrect predictions
6. **Debug errors** by examining failed predictions with LIME

### Important Differences from Neural Networks:

| Aspect | NN + LIME | RF + LIME |
|--------|----------|----------|
| **Feature scaling** | Required (StandardScaler) | Not needed |
| **Training time** | Slow (epochs, early stopping) | Fast (fit immediately) |
| **Interpretability** | Black-box (LIME helps) | Somewhat interpretable (trees) |
| **Feature importance** | Only through LIME/SHAP | Built-in + LIME/SHAP |
| **Speed** | Slower per prediction | Very fast per prediction |
| **Non-linearity** | Implicit (learned) | Explicit (tree splits) |
| **LIME perturbations** | More complex interactions | Simpler (tree logic) |

### Files Generated:

- `lime_rf_feature_importance_builtin.png` - RandomForest built-in importance
- `lime_rf_explanation_sample_*.png` - Individual LIME explanations
- `lime_rf_feature_importance.png` - LIME aggregated importance
- `lime_rf_impact_distributions.png` - Feature impact patterns
- `lime_rf_correct_vs_incorrect.png` - Prediction quality analysis

### Practical Example:

**LIME Explanation for RandomForest:**
```
Prediction: High Rating (0.87 probability)
  Route_LHR-JFK:    +0.18  (increases to high)
  Class_Business:   +0.22  (major positive factor)
  Verified_True:    +0.14  (boosts confidence)
  Traveller_Type_Couple:  -0.05  (slight negative)
```

This tells you: "The RandomForest predicted high rating because this was a verified Business class traveler on LHR-JFK route, which historically has high ratings."

### Advantages of RandomForest Over NN:

✓ **Simpler to interpret**: Tree structure is more transparent  
✓ **No hyperparameter tuning**: Works well with defaults  
✓ **Faster training**: No epochs or convergence issues  
✓ **Built-in importance**: Feature importance always available  
✓ **Handles mixed data**: No scaling needed  
✓ **More stable**: Less sensitive to data variations  
